==============================================================================
E/S/G 채점용 근거 검색 인터페이스 — 독립 실행 노트북

목적: 이미 구축된 Chroma DB에서, K-ESG 91개 항목(분류코드)별로 채점 근거
청크를 바로 뽑아주는 함수를 제공. E/S/G 담당자는 이 노트북의 함수만 쓰면 되고,
인덱싱 파이프라인 내부 구조를 몰라도 됨.

전제조건:
  1. build_rag_index_v5_2_pymupdf_only.ipynb (또는 그 이후 버전)로 Chroma DB가
     이미 구축돼 있어야 함 (Drive 백업 경로 동일하게 맞출 것)
  2. ESG_핵심지표_채점기준표_전체.xlsx (01_ESG통합진단마스터 시트)가 필요

⚠️ 현재 알려진 제약:
  - DB하이텍(000990)류처럼 pdfplumber+PyMuPDF 2단계로도 텍스트 추출이 심하게
    실패한 회사(지속가능경영보고서)는 그 문서 관련 근거가 비어있거나 부실할 수
    있음. 아래 [Cell 5]가 extraction_diagnostics.csv를 읽어서 이런 회사를
    자동으로 경고해줌 — LlamaParse 재처리(llamaparse_rescue.ipynb) 전까지는
    이 회사들의 채점 결과를 신뢰하지 말 것.
  - hybrid_weights([dense, bm25])는 아직 평가셋 튜닝 전 임시값([0.5, 0.5])
==============================================================================

==============================================================================
변경 이력

v1 (최초 배포):
  - get_evidence(), get_evidence_batch(), build_scoring_prompt() 정의
  - 회사별 텍스트 추출 실패율 경고 (extraction_diagnostics.csv 기반)

v1 -> v2:
  - [Cell 5b] list_available_companies() / sync_latest_from_drive() 추가
    (DB가 계속 커지는 중에도 "지금 몇 개사 준비됐는지" 확인 + 세션 재시작 없이 최신화)

v2 -> v3:
  - [Cell 4] load_rubric()에 header 자동 감지 추가. 전체 채점기준표
    (ESG_핵심지표_채점기준표_전체.xlsx)와 팀원별 분리본(E/S/G+P+반도체특화)이
    header 위치가 한 줄 어긋나는 문제 해결 — 이제 어떤 파일을 넣어도
    CONFIG["rubric_xlsx"] 경로 한 줄만 바꾸면 자동으로 맞게 읽힘

v3 -> v4:
  - [버그 수정] list_available_companies()가 연도(year)를 무시하고 회사당
    전체 doc_type을 한 리스트에 뭉쳐서, "리스트 길이==3"으로 완비 여부를
    잘못 판정하던 문제 수정. (company_code, year) 단위로 정확히 3종
    완비 여부를 체크하도록 변경. detail=True로 연도별 상세 조회 가능해짐
  - [주의: 하위 호환 깨짐] get_evidence(), get_evidence_batch(),
    build_scoring_prompt()에 year를 필수 인자로 추가.
    이유: company_code로만 필터링하면 같은 회사의 여러 연도 근거가 섞여서
    검색됨 (예: 2022년 채점 요청인데 2024년 보고서 수치가 잡힐 수 있음).
    기존 코드에서 get_evidence(company_code, item_code)처럼 쓰던 부분은
    get_evidence(company_code, year, item_code)로 인자 순서를 맞춰야 함

v4 -> v5 (GPT 리뷰 반영, 버그 3건 수정):
  - [치명적] FilteredHybridRetriever._get_bm25_retriever()의 캐시 키 생성 방식이
    tuple(sorted(dict.items()))였는데, 복합 필터(예: doc_type 지정, 혹은 v4에서
    year가 필수화되며 모든 호출이 해당됨)의 "$and": [...] 구조는 value가 list라서
    unhashable -> TypeError로 검색이 아예 안 됐음. json.dumps(sort_keys=True)로
    캐시 키를 문자열화하도록 수정 (v4 이후 사실상 모든 호출에서 발생하던 문제라 최우선 수정)
  - [Cell 5] extraction_diag_csv를 dtype 지정 없이 읽어서 "000990" 같은 기업코드가
    정수 990으로 읽히고, str(990)="990"이 되어 RISKY_COMPANIES 매칭이 항상 실패하던
    문제 수정 (dtype={"company_code": str} + zfill(6) 이중 방어)
  - [Cell 4] load_rubric()에 컬럼명 공백 제거 + '진단항목명'/'실사 진단 질문' NaN
    방어 추가 (전체/G 분리본은 문제 없음 확인했으나 E/S 분리본은 미확인이라 방어적으로 추가)

v5 -> v6 (GPT 리뷰 반영, 성능/견고성 개선 5건):
  - [Cell 3] BM25 캐시가 무제한으로 쌓이던 문제 -> FIFO 방식 크기 제한(기본 30) 추가.
    100개사 x 4개년 순회하면 캐시 항목마다 청크 데이터를 들고 있어 메모리가
    계속 증가할 수 있었음
  - [Cell 2, 3] candidate pool(dense/BM25 후보군, candidate_k=30)과 최종 반환
    개수(top_n, 기본 5)를 분리. 이전엔 둘이 같은 값(retrieval_k=10)을 썼는데,
    후보군을 넓게 뽑아야 정답이 top-k 밖으로 밀려나는 걸 줄일 수 있음
  - [Cell 6] get_evidence()에 company_code = str(company_code).zfill(6) 정규화 추가.
    RISKY_COMPANIES에는 이미 있던 방어인데 정작 핵심 함수엔 빠져있었음
  - [Cell 6] 같은 item_code의 쿼리 임베딩을 캐싱해서 재사용 (_query_embedding_cache).
    91개 항목 쿼리는 회사가 달라도 텍스트가 동일한데, 매번 재임베딩하면 API
    왕복이 100개사 x 91개 항목만큼 누적돼 전체 러닝타임에 영향을 줌
  - [Cell 5, 6] 위험도 체크를 회사 단위(RISKY_COMPANIES) -> (회사,연도,문서유형)
    단위(RISKY_DOCS)로 세분화. "지속가능경영보고서만 부실, 사업보고서는 정상"인
    회사가 통째로 위험 표시되던 문제 해결 — 실제로 반환된 근거의 출처 문서만 콕
    집어 경고하도록 변경

⚠️ 보류 중 (코드가 아니라 팀 확인이 필요한 사항):
  - metadata의 "year"가 정확히 무엇을 의미하는지(보고서 발행연도 vs 실제 성과연도)
    확인 필요. KCGS 평가년도가 전년도 실적 기준이었던 것처럼, 사업/지배구조/
    지속가능경영보고서도 발행연도와 다룬 회계연도가 다를 수 있음 — 파일명 규칙을
    정할 때 어느 쪽 기준으로 연도를 매겼는지 팀 내 확인 필요
==============================================================================


### [Cell 1] 설치 + Drive 마운트 + API 키

In [2]:
!pip install -q langchain langchain-google-genai langchain-chroma langchain-community \
    chromadb kiwipiepy rank_bm25 pandas openpyxl

from google.colab import drive
drive.mount('/content/drive')

import getpass
import os

os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google AI Studio API Key 입력 (임베딩 조회용): ")

Mounted at /content/drive
Google AI Studio API Key 입력 (임베딩 조회용): ··········


### [Cell 2] 설정값 — 메인 파이프라인과 경로 동일하게 맞출 것

In [3]:
DRIVE_ROOT = "/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/인덱스 결과"

CONFIG = {
    "embedding_model": "gemini-embedding-001",
    "persist_dir": "/content/chroma_db_local",  # 로컬 작업 (Drive 직접 접근 금지 — 손상 이력 있음)
    "drive_backup_dir": f"{DRIVE_ROOT}/chroma_db",
    "extraction_diag_csv": f"{DRIVE_ROOT}/extraction_diagnostics.csv",
    # [수정] 후보군(candidate pool)과 최종 반환 개수를 분리.
    # dense/BM25는 넓게(candidate_k) 뽑아서 RRF로 합친 뒤, 실제 LLM에는 top_n(기본 5)만 전달.
    "candidate_k": 30,
    "retrieval_k": 10,  # (하위 호환용, 지금은 candidate_k로 대체됨)
    "hybrid_weights": [0.5, 0.5],  # [dense, bm25] — 임시값, 평가셋 튜닝 전까지 유지

    # [담당자별로 이 한 줄만 바꾸면 됨] 채점기준표 경로.
    # 전체 파일(ESG_핵심지표_채점기준표_전체.xlsx)이든, 분리본
    # (..._E(환경).xlsx / ..._S(사회).xlsx / ..._G(지배구조)+P+반도체특화.xlsx)
    # 이든 아래 [Cell 4]의 load_rubric()이 알아서 인식함 — 파일 형식 신경 안 써도 됨.
    "rubric_xlsx": "/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/채점 시트/채점기준표/ESG_핵심지표_채점기준표_전체.xlsx",  # 예시: E 담당자
}

In [3]:
CONFIG["rubric_xlsx"] = "/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/채점 시트/채점기준표/ESG_핵심지표_채점기준표_G(지배구조)+P+반도체특화.xlsx"
# Cell 4 다시 실행해서 rubric_df 갱신

### [Cell 3] 라이브러리 import + Chroma 복원 + 하이브리드 리트리버 로드

In [4]:
import json
import shutil

import pandas as pd
from kiwipiepy import Kiwi
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings

kiwi = Kiwi()
RRF_K = 60

# BM25 품사 필터링 (조사/어미 노이즈 제거) — 명사/동사/형용사/외국어/숫자/한자만 사용
KIWI_KEEP_TAGS = {"NNG", "NNP", "NNB", "VV", "VA", "SL", "SN", "SH"}


def kiwi_tokenize(text: str):
    return [t.form for t in kiwi.tokenize(text) if t.tag in KIWI_KEEP_TAGS]


def restore_local_from_drive_backup():
    if os.path.exists(CONFIG["persist_dir"]):
        print("로컬 인덱스 이미 존재 (복원 스킵)")
        return
    if os.path.exists(CONFIG["drive_backup_dir"]):
        print("Drive 백업 -> 로컬 복원 중")
        shutil.copytree(CONFIG["drive_backup_dir"], CONFIG["persist_dir"])
    else:
        raise FileNotFoundError("Drive 백업이 없음 — 인덱싱 파이프라인을 먼저 돌릴 것")


class FilteredHybridRetriever:
    """메인 파이프라인과 동일한 구현 (BM25를 필터로 좁혀진 부분집합에서만 생성)."""

    def __init__(self, vectorstore, tokenize_func, k=10, weights=(0.5, 0.5), max_cache_entries=30):
        self.vectorstore = vectorstore
        self.tokenize_func = tokenize_func
        self.k = k  # dense/BM25 후보군 크기 (최종 반환 개수는 invoke()의 top_n으로 별도 지정)
        self.weights = weights
        # [수정] 캐시 무제한 증가 방지 — (회사, 연도) 조합이 100개사 x 4개년으로 늘면
        # 캐시 항목마다 회사-연도 전체 청크를 메모리에 들고 있어서 RAM이 계속 증가할 수 있음.
        # FIFO 방식으로 오래된 것부터 비움 (완전한 LRU는 아니지만 이 정도로 충분).
        self._bm25_cache: dict[str, object] = {}
        self._max_cache_entries = max_cache_entries

    def _get_bm25_retriever(self, filter_dict):
        # [수정] tuple(sorted(dict.items()))는 filter_dict에 "$and": [...]처럼
        # value가 list인 조건(복합 필터)이 섞이면 리스트가 unhashable이라
        # TypeError가 남. json.dumps(sort_keys=True)로 문자열화해서 캐시 키로 씀.
        sig = json.dumps(filter_dict, sort_keys=True, ensure_ascii=False)
        if sig in self._bm25_cache:
            return self._bm25_cache[sig]
        raw = self.vectorstore.get(where=filter_dict if filter_dict else None,
                                    include=["documents", "metadatas"])
        docs = [Document(page_content=d, metadata=m) for d, m in zip(raw["documents"], raw["metadatas"])]
        if not docs:
            self._set_cache(sig, None)
            return None
        retriever = BM25Retriever.from_documents(docs, preprocess_func=self.tokenize_func)
        retriever.k = self.k
        self._set_cache(sig, retriever)
        return retriever

    def _set_cache(self, sig, value):
        if len(self._bm25_cache) >= self._max_cache_entries:
            oldest_key = next(iter(self._bm25_cache))
            del self._bm25_cache[oldest_key]
        self._bm25_cache[sig] = value

    def invoke(self, query: str, filter: dict | None = None, top_n: int | None = None,
               query_embedding: list[float] | None = None):
        # [수정] query_embedding을 미리 계산해서 넘기면 dense 검색 시 재임베딩 API 호출을
        # 건너뜀 (같은 K-ESG 항목 쿼리를 회사마다 반복 임베딩하는 비효율 제거용)
        top_n = top_n or self.k
        if query_embedding is not None:
            dense_docs = self.vectorstore.similarity_search_by_vector(
                query_embedding, k=self.k, filter=filter
            )
        else:
            dense_docs = self.vectorstore.similarity_search(query, k=self.k, filter=filter)
        bm25_retriever = self._get_bm25_retriever(filter)
        bm25_docs = bm25_retriever.invoke(query) if bm25_retriever else []

        scores: dict[str, float] = {}
        lookup: dict[str, Document] = {}
        for rank, doc in enumerate(dense_docs):
            key = doc.metadata["chunk_id"]
            scores[key] = scores.get(key, 0.0) + self.weights[0] * (1.0 / (RRF_K + rank))
            lookup[key] = doc
        for rank, doc in enumerate(bm25_docs):
            key = doc.metadata["chunk_id"]
            scores[key] = scores.get(key, 0.0) + self.weights[1] * (1.0 / (RRF_K + rank))
            lookup[key] = doc

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return [lookup[k] for k, _ in ranked[:top_n]]

    def clear_cache(self):
        self._bm25_cache.clear()


restore_local_from_drive_backup()
embeddings = GoogleGenerativeAIEmbeddings(model=CONFIG["embedding_model"])
vectorstore = Chroma(persist_directory=CONFIG["persist_dir"], embedding_function=embeddings)
# [수정] k를 후보군(candidate pool) 크기로 사용 — dense/BM25 각각 넓게 뽑은 뒤
# RRF로 합치고, 실제로 LLM에 넣는 개수는 get_evidence()의 top_n(기본 5)이 따로 결정함.
hybrid_retriever = FilteredHybridRetriever(
    vectorstore=vectorstore, tokenize_func=kiwi_tokenize,
    k=CONFIG["candidate_k"], weights=CONFIG["hybrid_weights"],
)
print("Chroma DB + 하이브리드 리트리버 로드 완료")

/tmp/ipykernel_944/4257748404.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Drive 백업 -> 로컬 복원 중
Chroma DB + 하이브리드 리트리버 로드 완료


### [Cell 4] 채점기준표(91개 항목) 로드 — 분류코드로 바로 조회 가능하게

In [5]:
# [신규] 분리본 파일(E/S/G+P+반도체특화)은 맨 위에 "이건 분리본입니다" 설명 줄이
# 하나 더 있어서 header 위치가 전체 파일(header=0)과 한 줄 어긋남. '분류코드'
# 컬럼이 안 잡히면 header=1로 재시도 -> 전체/분리본 파일 아무거나 그대로 넣어도 됨.
def load_rubric(path: str, sheet_name: str = "01_ESG통합진단마스터") -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name=sheet_name)
    if "분류코드" not in df.columns:
        df = pd.read_excel(path, sheet_name=sheet_name, header=1)

    # [방어 코드] 확인된 전체/G 분리본엔 문제 없었지만, E/S 분리본은 미확인이라
    # 컬럼명 공백·검색 쿼리에 쓰이는 필드의 NaN을 방어적으로 처리해둠.
    # (NaN이 그대로 f-string에 들어가면 "온실가스 배출량 nan"처럼 쿼리가 오염됨)
    df.columns = [str(c).strip() for c in df.columns]
    for col in ("진단항목명", "실사 진단 질문"):
        if col in df.columns:
            df[col] = df[col].fillna("")
    return df


rubric_df = load_rubric(CONFIG["rubric_xlsx"])

# 분류코드(예: E-3-1) -> 행 전체를 바로 찾을 수 있게 인덱싱
rubric_by_code = rubric_df.set_index("분류코드").to_dict(orient="index")

print(f"항목 {len(rubric_df)}개 로드 완료")
print(rubric_df["영역"].value_counts())
print("\n예시 (G-1-1):")
print(rubric_by_code["G-1-1"])

항목 91개 로드 완료
영역
사회      34
환경      32
지배구조    20
정보공시     5
Name: count, dtype: int64

예시 (G-1-1):
{'No': 53, '적용구분': '기본진단항목(v2.0)', '영역': '지배구조', '범주': '이사회 구성', '진단항목명': '이사회 내 ESG 안건 상정', '항목설명': '\uf09f 조직의 최고의사결정기구인 이사회 또는 산하 위원회가 ESG 관련 안건을 보고받고 있으며, 특히 중대한 ESG 안건에 대해서는 심의/의결하고 있는지 확인 \uf09f 이사회 또는 산하 위원회를 통해 조직의 ESG 경영이 효율적으로 관리 및 감독되고 있는지 점검하기 위해 1) 활동내역, 2) 운영 규정의 2가지를 점검 \uf09f 조직은 ESG 안건을 다루기 위해 이사회 산하에 ‘지속가능경영위원회(ESG위원회)’를 설립할 수 있으나, 이사회, 또는 기존의 산하 위원회(사회책임위원회, 투명경영위원회, 거버넌스위원회, 경영위원회, 감사위원회 등)를 통해서도 ESG 안건을 다룰 수 있음', '데이터원천(수집처)': '이사회 또는 산하 위원회 개최 내역, 이사회 또는 산하 위원회 운영규정', '데이터기간': '직전 회계연도', '데이터범위': nan, '데이터산식(정량화공식)': nan, '채점유형': '복합가중형', '최대단계수': nan, '배점구조 요약': '복합가중형: 2~3개 세부요소를 균등가중평균(예: 1/2+1/2, 1/3×3) — 세부 배점은 점검기준 상세 참조, 자사취득점수는 수동 산정', '점검기준 상세(K-ESG v2.0 원문)': "영역1 (1/2) |  1단계 |  이사회 또는 산하 위원회에서 ESG 관련 안건을 다루고 있지 않는 경우 |  2단계 |  최근 1년간 ESG 관련 안건이 모두 '보고' 사항인 경우 |  3단계 |  최근 1년간 ESG 관련 안건 중 '심의/의결' 사항이 있는 경우 |  영역2 (1/2) |  1단계 |  이사회 또는 산하 위원회 운영규정에 ESG 관련 

### [Cell 5] 회사별 추출 품질 위험도 체크 — 채점 전 반드시 확인

In [6]:
# [수정] 회사 단위로만 위험도를 뭉치면 "지속가능경영보고서만 실패, 사업보고서는
# 정상"인 회사도 그냥 통째로 위험 표시됨. extraction_diagnostics.csv가 이미
# (company_code, year, doc_type) 단위 데이터를 갖고 있으니 그 단위로 그대로 씀.
RISKY_DOCS: dict[tuple, float] = {}  # {(company_code, year, doc_type): issue_ratio_after}
RISKY_COMPANIES: set[str] = set()    # list_available_companies() 요약용 (회사 단위 파생값)

if os.path.exists(CONFIG["extraction_diag_csv"]):
    # dtype 미지정 시 "000990" 같은 코드가 정수 990으로 읽혀서 앞자리 0이 소실됨
    # (str(990)="990" != "000990") -> 위험 경고 매칭 실패로 이어짐
    _diag = pd.read_csv(CONFIG["extraction_diag_csv"], dtype={"company_code": str})
    _risky = _diag[_diag["issue_ratio_after"] > 0.3]
    for _, r in _risky.iterrows():
        _code = str(r["company_code"]).zfill(6)  # 혹시 모를 소실 대비 추가 방어
        _year = int(r["year"])
        _doc_type = r["doc_type"]
        RISKY_DOCS[(_code, _year, _doc_type)] = r["issue_ratio_after"]
        RISKY_COMPANIES.add(_code)

    if RISKY_DOCS:
        print(f"⚠️ 텍스트 추출 실패율이 높은 문서 {len(RISKY_DOCS)}건 (회사/연도/문서유형 단위):")
        for (code, year, doc_type), ratio in sorted(RISKY_DOCS.items(), key=lambda x: -x[1]):
            print(f"  {code} / {year} / {doc_type}: 잔여 실패율 {ratio:.0%}")
    else:
        print("위험 문서 없음")
else:
    print("extraction_diagnostics.csv 없음 — 위험도 체크 생략 (경고 없이 진행됨에 주의)")


⚠️ 텍스트 추출 실패율이 높은 문서 20건 (회사/연도/문서유형 단위):
  061970 / 2023 / sustainability_report: 잔여 실패율 100%
  000990 / 2022 / sustainability_report: 잔여 실패율 100%
  060720 / 2023 / sustainability_report: 잔여 실패율 98%
  001440 / 2023 / sustainability_report: 잔여 실패율 95%
  001440 / 2024 / sustainability_report: 잔여 실패율 95%
  017040 / 2025 / sustainability_report: 잔여 실패율 95%
  011070 / 2024 / business_report: 잔여 실패율 95%
  001440 / 2022 / sustainability_report: 잔여 실패율 94%
  017040 / 2023 / sustainability_report: 잔여 실패율 94%
  017040 / 2024 / sustainability_report: 잔여 실패율 94%
  060720 / 2024 / sustainability_report: 잔여 실패율 91%
  000990 / 2025 / sustainability_report: 잔여 실패율 87%
  000990 / 2024 / sustainability_report: 잔여 실패율 87%
  000990 / 2023 / sustainability_report: 잔여 실패율 87%
  033640 / 2023 / sustainability_report: 잔여 실패율 73%
  033640 / 2025 / sustainability_report: 잔여 실패율 73%
  213420 / 2022 / sustainability_report: 잔여 실패율 51%
  213420 / 2024 / sustainability_report: 잔여 실패율 41%
  007340 / 2023 / sustaina

### [Cell 5b] (신규) 지금 몇 개사가 인덱싱 완료됐는지 확인 + 최신화
본작업(100개사)이 진행 중인 동안, 팀원은 매번 이 셀로 "지금 뭐가 준비돼 있는지"
확인하고 시작할 것. 세션을 새로 열지 않아도 `sync_latest_from_drive()`로
언제든 최신 상태로 갱신 가능.

In [11]:
CHECKPOINT_PATH = f"{DRIVE_ROOT}/checkpoint_completed.json"


REQUIRED_DOC_TYPES = {"business_report", "governance_report", "sustainability_report"}


def list_available_companies(detail: bool = False) -> pd.DataFrame:
    """체크포인트 로그 기준으로, 지금 시점에 인덱싱 완료된 회사/연도/문서 목록을 보여줌.

    [수정] 이전 버전은 연도를 무시하고 회사당 전체 doc_type을 한 리스트에 뭉쳐서,
    "리스트 길이==3"으로 완비 여부를 판정했음 — 이러면 여러 연도가 섞여 길이가
    3이 아니게 되거나(예: 4개년 x business만 완료 = 길이 4), 반대로 같은 문서유형이
    우연히 3번 나와도(예: 3개년 x business만) 잘못 '완비'로 뜨는 문제가 있었음.
    -> (company_code, year) 단위로 묶어서 3종이 실제로 다 있는지 정확히 체크하도록 수정.

    detail=True면 연도별 상세 테이블(어느 연도가 완비/부분/미비인지)까지 반환.
    """
    import json as _json

    if not os.path.exists(CHECKPOINT_PATH):
        print("체크포인트 파일 없음 — 아직 인덱싱이 시작 안 됐을 수 있음")
        return pd.DataFrame()

    with open(CHECKPOINT_PATH, encoding="utf-8") as f:
        checkpoint = _json.load(f)

    rows = []
    for doc_id in checkpoint:
        parts = doc_id.split("_")
        rows.append({
            "company_code": parts[0],
            "year": parts[1],
            "doc_type": "_".join(parts[2:]),
        })
    df = pd.DataFrame(rows)

    # (company_code, year) 단위로 실제 갖춰진 doc_type 집합 계산
    year_level = (
        df.groupby(["company_code", "year"])["doc_type"]
        .apply(lambda s: sorted(set(s)))
        .reset_index()
    )
    year_level["연도_3종_완비"] = year_level["doc_type"].apply(
        lambda types: set(types) == REQUIRED_DOC_TYPES
    )

    if detail:
        year_level["위험도_경고"] = year_level["company_code"].isin(RISKY_COMPANIES)
        return year_level.sort_values(["company_code", "year"])

    # 회사 단위 요약: 3종 완비된 연도가 1개 이상 있는지 + 몇 개 연도가 완비됐는지
    company_summary = (
        year_level.groupby("company_code")
        .agg(
            완비_연도수=("연도_3종_완비", "sum"),
            처리된_연도수=("year", "nunique"),
            완비_연도목록=("year", lambda s: sorted(
                y for y, ok in zip(s, year_level.loc[s.index, "연도_3종_완비"]) if ok
            )),
        )
        .reset_index()
    )
    company_summary["위험도_경고"] = company_summary["company_code"].isin(RISKY_COMPANIES)

    n_ready = (company_summary["완비_연도수"] > 0).sum()
    print(f"인덱싱된 회사: {df['company_code'].nunique()}개 "
          f"(3종 전부 완비된 연도가 1개 이상 있는 회사: {n_ready}개)")
    print("연도별 상세가 필요하면 list_available_companies(detail=True) 호출할 것")
    return company_summary


def sync_latest_from_drive():
    """세션 중간에 인덱싱이 더 진행됐을 때, 재시작 없이 최신 Drive 백업으로 갱신.
    (restore_local_from_drive_backup은 로컬에 이미 있으면 스킵하므로, 최신화하려면
    이 함수를 명시적으로 호출해야 함 — 로컬을 지우고 다시 복사.)"""
    global vectorstore, hybrid_retriever
    if os.path.exists(CONFIG["persist_dir"]):
        shutil.rmtree(CONFIG["persist_dir"])
    restore_local_from_drive_backup()
    vectorstore = Chroma(persist_directory=CONFIG["persist_dir"], embedding_function=embeddings)
    hybrid_retriever = FilteredHybridRetriever(
        vectorstore=vectorstore, tokenize_func=kiwi_tokenize,
        k=CONFIG["retrieval_k"], weights=CONFIG["hybrid_weights"],
    )
    print("최신 인덱스로 갱신 완료")


list_available_companies()


인덱싱된 회사: 45개 (3종 전부 완비된 연도가 1개 이상 있는 회사: 31개)
연도별 상세가 필요하면 list_available_companies(detail=True) 호출할 것


,company_code,완비_연도수,처리된_연도수,완비_연도목록,위험도_경고
0,000500,2,4,"[2024, 2025]",False
1,000660,4,4,"[2022, 2023, 2024, 2025]",False
2,000990,4,4,"[2022, 2023, 2024, 2025]",True
3,001440,4,4,"[2022, 2023, 2024, 2025]",True
4,004490,4,4,"[2022, 2023, 2024, 2025]",False
5,005680,0,2,[],False
6,005930,4,4,"[2022, 2023, 2024, 2025]",False
7,006400,4,4,"[2022, 2023, 2024, 2025]",False
8,007340,3,4,"[2023, 2024, 2025]",True
9,007660,1,3,[2024],False


### [Cell 6] 핵심 함수 — get_evidence(), get_evidence_batch(), build_scoring_prompt()

In [8]:
_query_embedding_cache: dict[str, list[float]] = {}  # [신규] item_code -> 쿼리 임베딩 캐시


def get_evidence(company_code: str, year: int, item_code: str, top_n: int = 10,
                  doc_type: str | None = None) -> list[dict]:
    """K-ESG 91개 항목 중 하나(item_code, 예: 'E-3-1')에 대한 채점 근거 청크를 검색.

    [수정] year를 필수 인자로 추가. 같은 회사라도 연도마다 보고서 내용(배출량 수치,
    이사회 구성 등)이 다른데, company_code로만 필터링하면 검색 결과에 여러 연도
    근거가 섞여 나옴 — 예: 2022년 채점인데 2024년 보고서 수치가 잡힐 수 있음.
    반드시 채점 대상 연도를 명시할 것 (선택 인자로 안 둔 이유: 깜빡하고 안 넣으면
    여러 연도가 섞인 채로 조용히 검색되는 실수를 원천 차단하기 위함).

    쿼리는 채점기준표의 '진단항목명' + '실사 진단 질문'을 그대로 씀 —
    항목 라벨(짧은 키워드)과 실제 질문(자연어)을 합쳐서 dense/BM25 둘 다 잘 걸리게 함.

    반환: [{"doc_type":..., "page":..., "section_title":..., "content":...}, ...]
    """
    company_code = str(company_code).zfill(6)  # [수정] "660"/660처럼 넣어도 안전하게

    if item_code not in rubric_by_code:
        raise KeyError(f"채점기준표에 없는 분류코드: {item_code}")

    item = rubric_by_code[item_code]
    query = f"{item['진단항목명']} {item['실사 진단 질문']}"

    conditions = [{"company_code": company_code}, {"year": year}]
    if doc_type:
        conditions.append({"doc_type": doc_type})
    filter_dict = {"$and": conditions}

    # [수정] 같은 item_code 쿼리는 회사가 달라도 텍스트가 동일 -> 쿼리 임베딩을
    # 캐싱해서 재사용 (100개사 x 91항목 = 매번 재임베딩하면 API 왕복이 누적돼 느려짐)
    if item_code not in _query_embedding_cache:
        _query_embedding_cache[item_code] = embeddings.embed_query(query)
    query_embedding = _query_embedding_cache[item_code]

    results = hybrid_retriever.invoke(
        query, filter=filter_dict, top_n=top_n, query_embedding=query_embedding
    )
    if not results:
        print(f"⚠️ {company_code} / {year}년 근거 0건 — list_available_companies(detail=True)로 "
              f"이 회사·연도가 실제로 인덱싱됐는지 먼저 확인할 것")

    # [수정] 회사 단위가 아니라, 실제로 반환된 근거의 (연도, 문서유형)이 위험
    # 목록에 있는지 개별 확인 — "지속가능경영보고서만 부실, 사업보고서는 정상"인
    # 경우를 구분해서 알려줌
    _flagged_doc_types = set()
    for d in results:
        key = (company_code, year, d.metadata.get("doc_type"))
        if key in RISKY_DOCS and d.metadata.get("doc_type") not in _flagged_doc_types:
            _flagged_doc_types.add(d.metadata.get("doc_type"))
            print(f"⚠️ {company_code} / {year} / {d.metadata.get('doc_type')}: "
                  f"추출 실패율 {RISKY_DOCS[key]:.0%} — 이 문서에서 나온 근거는 신뢰도 낮음")

    return [
        {
            "doc_type": d.metadata.get("doc_type"),
            "page": d.metadata.get("page"),
            "section_title": d.metadata.get("section_title"),
            "content": d.page_content,
        }
        for d in results
    ]


def get_evidence_batch(company_code: str, year: int, area: str, top_n: int = 10) -> dict[str, list[dict]]:
    """영역(area) 하나에 속한 모든 항목의 근거를 한 번에 조회 (특정 연도 기준).
    area 값: '환경'(E, 32개) / '사회'(S, 34개) / '지배구조'(G, 20개) / '정보공시'(5개)
    반환: {item_code: [evidence, ...], ...}
    """
    codes = rubric_df[rubric_df["영역"] == area]["분류코드"].tolist()
    if not codes:
        raise ValueError(f"'{area}'에 해당하는 항목 없음. 사용 가능한 값: "
                          f"{rubric_df['영역'].unique().tolist()}")
    print(f"{company_code} / {year}년 / {area} 영역 {len(codes)}개 항목 조회 중...")
    return {code: get_evidence(company_code, year, code, top_n=top_n) for code in codes}


def build_scoring_prompt(company_code: str, year: int, item_code: str, evidence: list[dict]) -> str:
    """LLM 채점 호출에 바로 넣을 수 있는 프롬프트 뼈대.
    E/S/G 담당자가 이 뼈대에 본인 프롬프트 스타일/출력 스키마를 얹어서 쓰면 됨."""
    item = rubric_by_code[item_code]
    evidence_text = "\n\n".join(
        f"[근거 {i+1}] ({e['doc_type']} p.{e['page']}, 섹션: {e['section_title'] or '미상'})\n{e['content']}"
        for i, e in enumerate(evidence)
    ) or "(검색된 근거 없음)"

    return f"""다음은 K-ESG 진단항목 채점을 위한 정보입니다.

[대상]
- 기업코드: {company_code} / 채점 연도: {year} (해당 연도 보고서 기준으로만 채점할 것)

[항목 정보]
- 분류코드: {item_code}
- 영역/범주: {item['영역']} / {item['범주']}
- 진단항목명: {item['진단항목명']}
- 항목설명: {item['항목설명']}
- 채점유형: {item['채점유형']} (답변형식: {item['답변형식']})
- 점검기준: {item['점검기준 상세(K-ESG v2.0 원문)']}
- 실사 진단 질문: {item['실사 진단 질문']}

[검색된 근거]
{evidence_text}

위 근거만을 바탕으로 이 항목을 채점하세요. 근거가 불충분하면 "근거부족"이라고 답하세요.
"""


print("get_evidence(), get_evidence_batch(), build_scoring_prompt() 정의 완료")

get_evidence(), get_evidence_batch(), build_scoring_prompt() 정의 완료


In [13]:
# 회사별 텍스트 추출 실패율 및 성공율 전체 집계
diag_path = CONFIG["extraction_diag_csv"]

if os.path.exists(diag_path):
    df_diag = pd.read_csv(diag_path, dtype={"company_code": str})
    df_diag["company_code"] = df_diag["company_code"].str.zfill(6)

    # 1. 회사별 집계 (평균 실패율, 최대 실패율, 고위험 문서 건수 등)
    summary = df_diag.groupby("company_code").agg(
        총_문서수=("doc_type", "count"),
        평균_실패율=("issue_ratio_after", "mean"),
        최대_실패율=("issue_ratio_after", "max"),
        고위험_문서수=("issue_ratio_after", lambda x: (x > 0.3).sum())
    ).reset_index()

    # 2. 성공율 계산 (1 - 평균 실패율)
    summary["평균_성공율"] = 1.0 - summary["평균_실패율"]

    # 3. 정렬 (실패율 높은 순)
    summary = summary.sort_values(by="평균_실패율", ascending=False)

    # 퍼센트 포맷 적용하여 출력용 데이터프레임 생성
    display_df = summary.copy()
    display_df["평균_성공율"] = display_df["평균_성공율"].apply(lambda x: f"{x:.1%}")
    display_df["평균_실패율"] = display_df["평균_실패율"].apply(lambda x: f"{x:.1%}")
    display_df["최대_실패율"] = display_df["최대_실패율"].apply(lambda x: f"{x:.1%}")

    print(f"총 {len(summary)}개 회사 전체 품질 집계 (실패율 높은 순):\n")
    print(display_df.to_string(index=False))
else:
    print(f"'{diag_path}' 파일이 존재하지 않습니다.")

총 27개 회사 전체 품질 집계 (실패율 높은 순):

company_code  총_문서수 평균_실패율 최대_실패율  고위험_문서수 평균_성공율
      060720      4  47.8%  98.1%        2  52.2%
      017040      7  40.9%  94.7%        3  59.1%
      000990     12  30.8% 100.0%        4  69.2%
      033640      5  29.7%  73.2%        2  70.3%
      001440     12  26.0%  95.2%        3  74.0%
      061970      4  25.3% 100.0%        1  74.7%
      213420      8  16.3%  51.2%        2  83.7%
      011070     10  10.9%  94.6%        1  89.1%
      090460      6   7.7%  25.9%        0  92.3%
      336260     11   5.7%  34.7%        1  94.3%
      108320     11   5.4%  29.1%        0  94.6%
      192650      5   4.3%  11.8%        0  95.7%
      009450      9   4.2%  22.0%        0  95.8%
      007340     11   3.8%  34.9%        1  96.2%
      000500     11   2.6%   9.9%        0  97.4%
      020150     10   1.9%   8.8%        0  98.1%
      034220     12   1.7%   7.5%        0  98.3%
      393890      2   1.6%   2.5%        0  98.4%
      322000      8

### [Cell 7] 사용 예시

In [30]:
# 예시 1: 항목 1개 근거 조회 (SK하이닉스, 2022년, 온실가스 배출량)
# [수정] year 필수 인자 추가 — 채점 대상 연도를 항상 명시할 것
evidence = get_evidence("000660", 2022, "G-1-1", top_n=5)
for e in evidence:
    print(f"[{e['doc_type']} p.{e['page']}] {e['content'][:100]}...")

print("\n" + "=" * 60)

# 예시 2: 프롬프트 뼈대까지 완성된 형태로
prompt = build_scoring_prompt("000660", 2022, "G-1-1", evidence)
print(prompt[:800])

# 예시 3: 환경(E) 영역 32개 항목 한 번에 (담당자별로 이렇게 시작하면 됨)
# all_E_evidence = get_evidence_batch("000660", 2022, "환경")
# all_S_evidence = get_evidence_batch("000660", 2022, "사회")
# all_G_evidence = get_evidence_batch("000660", 2022, "지배구조")

# 예시 4: 같은 회사, 여러 연도 순회 (KCGS 등급 lag 피처처럼 연도별로 다 필요할 때)
# for y in [2022, 2023, 2024, 2025]:
#     yearly_evidence = get_evidence_batch("000660", y, "환경")

[sustainability_report p.108] |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  | 108 |
|---|---|---|---|---|---|---|---|---|-...
[sustainability_report p.33] 33
Introduction ESG Strategy Pursue R I S M ESG Data Appendix
ESG 경영을 위한 적극적인 리더십, ESG경영위원회
SK하이닉스는 ...
[sustainability_report p.108] |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  | 108 |
|---|---|---|---|---|---|---|---|---|-...
[business_report p.242] | 위 임 | - 다음 각호의 사항을 제외하고는 그 권한을 이사회내 위원회에 위임할 수 있음 1. 주주총회 승인을 요하는 사항의 제안 2. 대표이사의 선임 및 해임 3. 위원회 설...
[business_report p.241] | 구 성 | 주주총회에서 선임된 이사 전원으로 구성함 |  |
|---|---|---|
| 개최시기 | 정기 이사회 | 매분기 종료 후 45일 내에 개최함 |
|  | 임시 이사...

다음은 K-ESG 진단항목 채점을 위한 정보입니다.

[대상]
- 기업코드: 000660 / 채점 연도: 2022 (해당 연도 보고서 기준으로만 채점할 것)

[항목 정보]
- 분류코드: G-1-1
- 영역/범주: 지배구조 / 이사회 구성
- 진단항목명: 이사회 내 ESG 안건 상정
- 항목설명:  조직의 최고의사결정기구인 이사회 또는 산하 위원회가 ESG 관련 안건을 보고받고 있으며, 특히 중대한 ESG 안건에 대해서는 심의/의결하고 있는지 확인  이사회 또는 산하 위원회를 통해 조직의 ESG 경영이 효율적으로 관리 및 감독되고 있는지 점검하기 위해 1) 활동내역, 2) 운영 규정의 2가지를 점검 

In [18]:
list_available_companies(detail=True)

,company_code,year,doc_type,연도_3종_완비,위험도_경고
0,000500,2022,"[business_report, sustainability_report]",False,False
1,000500,2023,"[business_report, sustainability_report]",False,False
2,000500,2024,"[business_report, governance_report, sustainab...",True,False
3,000500,2025,"[business_report, governance_report, sustainab...",True,False
4,000660,2022,"[business_report, governance_report, sustainab...",True,False
...,...,...,...,...,...
122,373220,2024,"[business_report, governance_report, sustainab...",True,False
123,373220,2025,"[business_report, governance_report, sustainab...",True,False
124,393890,2024,[business_report],False,False
125,393890,2025,[sustainability_report],False,False


In [10]:
from pydantic import BaseModel
from typing import Literal
from langchain_google_genai import ChatGoogleGenerativeAI

class ESGItemScore(BaseModel):
    item_code: str
    score: str            # 답변형식에 맞게 (단계 1~5, O/X 등)
    rationale: str
    evidence_page: int | None
    confidence: Literal["High", "Medium", "Low"]

class ESGItemScoreList(BaseModel):
    scores: list[ESGItemScore]

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)
structured_llm = llm.with_structured_output(ESGItemScoreList)


def score_area(company_code: str, year: int, area: str, top_n: int = 5) -> list[dict]:
    """영역 하나(지배구조/정보공시/환경) 전체를 한 번의 LLM 호출로 채점."""
    codes = rubric_df[rubric_df["영역"] == area]["분류코드"].tolist()
    evidence_batch = get_evidence_batch(company_code, year, area, top_n=top_n)

    context_blocks = []
    for code in codes:
        item = rubric_by_code[code]
        ev_text = "\n".join(
            f"  - ({e['doc_type']} p.{e['page']}) {e['content'][:500]}"
            for e in evidence_batch[code]
        ) or "  (근거 없음)"
        context_blocks.append(
            f"[{code}] {item['진단항목명']}\n"
            f"질문: {item['실사 진단 질문']}\n"
            f"답변형식: {item['답변형식']}\n"
            f"근거:\n{ev_text}"
        )

    prompt = (
        f"다음은 {company_code}의 {year}년 {area} 영역 K-ESG 채점 대상 항목들입니다.\n"
        f"각 항목을 근거에 기반해서만 채점하세요. 근거가 불충분하면 confidence를 "
        f"Low로 표시하고, 임의로 점수를 추정하지 마세요.\n\n"
        + "\n\n".join(context_blocks)
    )

    result = structured_llm.invoke(prompt)
    return [
        s.model_dump() | {"company_code": company_code, "year": year, "area": area}
        for s in result.scores
    ]

In [11]:
# SK하이닉스 2022년 '지배구조' 영역만 1회 테스트 실행
test_scores = score_area("000660", 2022, "지배구조")

# 결과 샘플 확인
test_df = pd.DataFrame(test_scores)
print(test_df[["item_code", "score", "confidence", "rationale"]].head())

000660 / 2022년 / 지배구조 영역 20개 항목 조회 중...


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


  item_code score confidence  \
0     G-1-1     5       High   
1     G-1-2     5       High   
2     G-1-3     5       High   
3     G-1-4     3       High   
4     G-1-5     3       High   

                                           rationale  
0  ESG경영위원회 및 지속경영위원회를 통하여 핵심 ESG 안건 및 기후변화 대응 이슈...  
1     이사회 총 9명 중 사외이사 6명으로 66.7%의 과반수 비율을 충족하고 있습니다.  
2  대표이사와 이사회 의장을 분리하고 사외이사를 이사회 의장으로 선임하여 관련 현황을 ...  
3    여성 사외이사 1명(한애라 이사)을 확보하여 이사회 성별 다양성을 유지하고 있습니다.  
4  반도체 소자 및 회로 기술 전문가인 신창환 교수(성균관대 전자전기공학부)가 사외이사...  


In [15]:
print(test_df[["item_code", "score", "confidence", "rationale"]].to_string())

     item_code score confidence                                                                         rationale
0        G-1-1     5       High  ESG경영위원회 및 지속경영위원회를 통하여 핵심 ESG 안건 및 기후변화 대응 이슈를 정기적으로 논의하고 이사회에도 상정 및 보고하고 있습니다.
1        G-1-2     5       High                                    이사회 총 9명 중 사외이사 6명으로 66.7%의 과반수 비율을 충족하고 있습니다.
2        G-1-3     5       High                          대표이사와 이사회 의장을 분리하고 사외이사를 이사회 의장으로 선임하여 관련 현황을 공시하고 있습니다.
3        G-1-4     3       High                                   여성 사외이사 1명(한애라 이사)을 확보하여 이사회 성별 다양성을 유지하고 있습니다.
4        G-1-5     3       High                        반도체 소자 및 회로 기술 전문가인 신창환 교수(성균관대 전자전기공학부)가 사외이사로 포함되어 있습니다.
5        G-2-1     5       High                                       이사회 평균 참석률이 98% 이상으로 기준인 75% 이상을 상회하고 있습니다.
6        G-2-2     5       High                                                      사내이사의 이사회 참석률이 높게 유지되고 있습니다.
7        G-2-3     5       High             감사위원회, 사외이사후보추천위원회, 지속경영위원회, 인사보상위원회, 미래전략위원

In [26]:
ready = list_available_companies(detail=True)
ready = ready[ready["연도_3종_완비"]]

all_scores = []
for _, row in ready.iterrows():
    for area in ["지배구조", "정보공시", "환경"]:
        all_scores.extend(score_area(row["company_code"], int(row["year"]), area))

results_df = pd.DataFrame(all_scores)
results_df["need_review"] = results_df["confidence"] == "Low"  # 0점 대신 검토 대상으로 플래그

results_df.to_csv(".../G_scores_draft.csv", index=False, encoding="utf-8-sig")
print(f"채점 완료: {len(results_df)}건 (need_review {results_df['need_review'].sum()}건)")

NameError: name 'get_evidence_batch' is not defined